# Artifact Assessor — Inference Demo

Run the fine-tuned Gemma 4 E2B LoRA adapter on any image.  
**Kernel:** `gemma4_finetune` conda env  
**Prereq:** LoRA adapter exists at `./outputs/google_gemma-4-E2B-it_artifact_assessor_lora/`

In [ ]:
import sys, os
# Allow importing from scripts/ regardless of where the notebook is opened
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(repo_root, 'scripts'))
os.chdir(repo_root)  # makes relative paths (./outputs, ./data) resolve correctly

import torch
from peft import PeftModel
from transformers import AutoProcessor, Gemma4ForConditionalGeneration

from utils import ADAPTER_PATH, DEVICE, DTYPE, MODEL_ID, USER_PROMPT

print(f'Device : {DEVICE} | dtype : {DTYPE}')
print(f'Adapter: {ADAPTER_PATH}')

## 1 — Load Model

Loads the base Gemma 4 E2B weights, then applies the saved LoRA adapter.  
Takes ~30–60 s on first run (downloads weights from HF Hub if not cached).

In [ ]:
print('Loading processor...')
processor = AutoProcessor.from_pretrained(ADAPTER_PATH)

print(f'Loading base model {MODEL_ID}...')
base = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto' if DEVICE == 'cuda' else None,
    attn_implementation='sdpa',
)

print('Applying LoRA adapter...')
model = PeftModel.from_pretrained(base, ADAPTER_PATH)
if DEVICE != 'cuda':
    model = model.to(DEVICE)
model.eval()

print('Ready.')

## 2 — Inference Helper

In [ ]:
from PIL import Image

@torch.no_grad()
def assess_artifacts(
    image: Image.Image,
    max_new_tokens: int = 192,
    prompt: str = USER_PROMPT,
) -> str:
    """Return a natural-language artifact description for image."""
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': prompt},
        ],
    }]
    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = processor(
        text=text,
        images=[[image]],
        return_tensors='pt',
    ).to(DEVICE)

    out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = out_ids[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True)

## 3 — Option A: Run on a Custom Image

Change `IMAGE_PATH` to any local file.

In [ ]:
import matplotlib.pyplot as plt

IMAGE_PATH = '/path/to/your/image.jpg'  # <-- change this

image = Image.open(IMAGE_PATH).convert('RGB')

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis('off')
plt.title('Input image')
plt.show()

result = assess_artifacts(image)
print('Artifact assessment:\n')
print(result)

## 3 — Option B: Run on Validation Set Samples

Requires `./data/vqa_val` to exist (output of script 02).

In [ ]:
from datasets import load_from_disk

val_ds = load_from_disk('./data/vqa_val')
print(f'Validation samples: {len(val_ds):,}')
print('Columns:', val_ds.column_names)

In [ ]:
# Change SAMPLE_IDX to inspect different examples
SAMPLE_IDX = 0

sample = val_ds[SAMPLE_IDX]
image  = sample['image']
gt     = sample['assistant_response']

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis('off')
plt.title(f'Validation sample #{SAMPLE_IDX}')
plt.show()

pred = assess_artifacts(image)

print('Ground truth:\n', gt)
print()
print('Model output:\n', pred)

## 4 — Browse Multiple Samples

Iterates over `N` validation samples and prints results side by side.

In [ ]:
N = 5  # number of samples to show

fig, axes = plt.subplots(1, N, figsize=(4 * N, 4))
if N == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.imshow(val_ds[i]['image'])
    ax.axis('off')
    ax.set_title(f'#{i}')
plt.tight_layout()
plt.show()

for i in range(N):
    sample = val_ds[i]
    pred = assess_artifacts(sample['image'])
    print(f'--- Sample #{i} ---')
    print('GT  :', sample['assistant_response'])
    print('Pred:', pred)
    print()

## 5 — Interactive Widget (optional)

Upload an image directly in the notebook. Requires `ipywidgets`.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import io

    uploader = widgets.FileUpload(accept='image/*', multiple=False)
    out      = widgets.Output()

    def on_upload(change):
        with out:
            clear_output()
            content = next(iter(uploader.value.values()))['content']
            img = Image.open(io.BytesIO(content)).convert('RGB')
            plt.figure(figsize=(5, 5))
            plt.imshow(img)
            plt.axis('off')
            plt.show()
            print('Running...')
            print(assess_artifacts(img))

    uploader.observe(on_upload, names='value')
    display(uploader, out)

except ImportError:
    print('ipywidgets not installed. Run: pip install ipywidgets')